In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TaxiDataAnalysis") \
    .getOrCreate()

In [4]:
input_path = "/workspaces/Spark_CodeSpace/data/raw/yellow_taxi/yellow_tripdata_2025-11.parquet"
df = spark.read.parquet(input_path)

In [ ]:
#Q1
df.repartition(4).write.parquet("/workspaces/Spark_CodeSpace/data/processed/yellow_taxi/")

In [7]:
#Q2
df_15_nov = df.where(col("tpep_pickup_datetime").startswith("2025-11-15"))
df_15_nov.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-15 00:00:16|  2025-11-15 00:07:22|              2|         1.27|         1|                 N|         137|    

In [49]:
#Q4
from pyspark.sql import functions as F
df_duration = df.withColumn("duration_hrs", 
    ((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600)
)
df_duration.select(F.round(F.max("duration_hrs"), 2).alias("max_duration_hrs")).show()

+----------------+
|max_duration_hrs|
+----------------+
|           90.65|
+----------------+



In [35]:
#Q5
df_csv = spark.read.csv("/workspaces/Spark_CodeSpace/data/raw/yellow_taxi/taxi_zone_lookup.csv", header=True, inferSchema=True)

In [44]:
df_join = df_csv.join(df, df_csv["LocationID"] == df["PULocationID"], "inner")
df_join.createOrReplaceTempView("taxi_data")
spark.sql('''
          SELECT PULocationID,Zone,count(*) as trip_count FROM taxi_data
          GROUP BY PULocationID,Zone
          order BY trip_count asc
        ''').show(5,truncate=False)



+------------+---------------------------------------------+----------+
|PULocationID|Zone                                         |trip_count|
+------------+---------------------------------------------+----------+
|105         |Governor's Island/Ellis Island/Liberty Island|1         |
|5           |Arden Heights                                |1         |
|84          |Eltingville/Annadale/Prince's Bay            |1         |
|187         |Port Richmond                                |3         |
|204         |Rossville/Woodrow                            |4         |
+------------+---------------------------------------------+----------+
only showing top 5 rows
